<p align="left">
  <a href="https://colab.research.google.com/github/wgomezf/CNN_workshop/blob/main/Notebooks/LeNet5_test.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200">
  </a>
</p>

# Handwritten digit recognition using the LeNet-5 model

This example tests the LeNet-5 model trained on the MNIST database with your own handwritten digits.

In [ ]:
#@title Load necessary libraries
import tensorflow as tf
from google.colab import drive
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

In [ ]:
#@title Import the LeNet-5 model from Google drive
#This model has to be trained in the LeNet-5 practice
drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/lenet5.keras'
lenet5 = tf.keras.models.load_model(model_path)

In [ ]:
#@title Canvas to draw a digit

canvas_html = """
<canvas id="canvas" width="256" height="256" style="border:2px solid black; cursor:crosshair;"></canvas>
<br>
<button id="save-btn" style="margin-top: 10px; padding: 5px 15px;">Save Drawing</button>
<script>
  var canvas = document.getElementById('canvas');
  var ctx = canvas.getContext('2d');
  ctx.fillStyle = "black"; // Set background color
  ctx.fillRect(0, 0, canvas.width, canvas.height);

  var drawing = false;
  var mousePos = {x: 0, y: 0};

  canvas.addEventListener('mousedown', function(e) {
    drawing = true;
    mousePos = getMousePos(canvas, e);
  });

  canvas.addEventListener('mouseup', function(e) {
    drawing = false;
  });

  canvas.addEventListener('mousemove', function(e) {
    if (drawing) {
      var pos = getMousePos(canvas, e);
      drawLine(ctx, mousePos, pos);
      mousePos = pos;
    }
  });

  function getMousePos(canvas, evt) {
    var rect = canvas.getBoundingClientRect();
    return {
      x: evt.clientX - rect.left,
      y: evt.clientY - rect.top
    };
  }

  function drawLine(ctx, from, to) {
    ctx.beginPath();
    ctx.moveTo(from.x, from.y);
    ctx.lineTo(to.x, to.y);
    ctx.strokeStyle = 'white';
    ctx.lineWidth = 20;
    ctx.lineJoin = 'round';
    ctx.lineCap = 'round';
    ctx.stroke();
  }

  var dataPromise = new Promise(resolve => {
    document.getElementById('save-btn').onclick = function() {
      var data = canvas.toDataURL('image/png');
      resolve(data);
    };
  });
</script>
"""

display(HTML(canvas_html))

# Wait for the user to click the "Save Drawing" button and get the data
print("Draw in the box above and click 'Save Drawing' when finished.")
result_data = eval_js("dataPromise")


In [ ]:
#@title Classify hadwritten digit with LeNet-5 model

# Extract base64 image data and convert to bytes
image_data = result_data.split(',')[1]
binary_data = b64decode(image_data)

# Open as a PIL Image
img = Image.open(io.BytesIO(binary_data))

# Adjust image for classification
target_size = (28, 28) # CNN input size
img = img.convert('L') # Convert to grayscale
img = img.resize(target_size, Image.NEAREST)
img = image.img_to_array(img) # Convert to numpy array
img /= 255.0 # Normalize pixels to 0-1 range
I = img.copy() # Save a copy for later

# Classify handwritten digit
lbs = [str(i) for i in range(10)]
img = np.expand_dims(img, axis=0) # Add batch dimension
y_prob = lenet5.predict(img, verbose=0) # Predict with LeNet-5
y_pred = np.argmax(y_prob, axis=1)

# Show image
plt.figure(figsize=(8, 4)) # Create a figure to hold both subplots
plt.subplot(1, 2, 1)
plt.imshow(I, cmap='gray')
plt.axis('off')
plt.title(f'Predicted digit: {lbs[y_pred[0]]}')

plt.subplot(1, 2, 2)
plt.bar(range(10), y_prob[0], color='skyblue', edgecolor='black')
plt.xticks(range(10), lbs) # Set x-axis ticks to be the digit labels
plt.xlabel('Digit')
plt.ylabel('Probability')
plt.title('Predicted probabilities')
plt.show()
